# IMDB Sentiment Analysis: full pipeline on Colab

Runs every experiment in this repo end to end. Before starting, switch to a GPU: **Runtime → Change runtime type → T4 GPU**.

Rough timings on a free T4: baseline ~3 min, DistilBERT (256 tokens) ~25 min, RoBERTa-base (512 tokens) ~1.5–2 h.
Results (metrics, figures, error reports) land in `results/`. Commit that folder back to the repo afterwards.

In [ ]:
!git clone https://github.com/haikalfr04/Sentiment-Analysis.git
%cd Sentiment-Analysis
!pip install -q -r requirements.txt
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Exploratory data analysis

In [ ]:
!python -m src.eda --tokenizer distilbert-base-uncased
from IPython.display import Image
Image("results/figures/length_distribution.png")

## 2. Baseline: TF-IDF + Logistic Regression

In [ ]:
!python -m src.baseline

## 3. Fine-tune transformers

The first two runs are a truncation ablation: same model and budget, only *which* 256 tokens are kept changes.

In [ ]:
!python -m src.train --model_name distilbert-base-uncased --max_length 256 --truncation head

In [ ]:
!python -m src.train --model_name distilbert-base-uncased --max_length 256 --truncation head_tail

Strongest setup. Batch 8 with gradient accumulation 2 (effective batch 16) keeps 512-token RoBERTa within T4 memory.

In [ ]:
!python -m src.train --model_name roberta-base --max_length 512 --truncation head_tail --batch_size 8 --grad_accum 2

## 4. Compare runs and analyse errors

In [ ]:
!python -m src.evaluate
from IPython.display import Markdown, display
display(Markdown(open("results/comparison.md").read()))

In [ ]:
import glob
from IPython.display import Image, display
for path in sorted(glob.glob("results/figures/*_confusion.png")):
    display(Image(path, width=350))

In [ ]:
from IPython.display import Markdown, display
display(Markdown(open("results/errors/roberta-base-512-head_tail_errors.md").read()))

## 5. (Optional) Push the best model to the Hugging Face Hub

Needed for the Gradio demo on Hugging Face Spaces. Create a *write* token at https://huggingface.co/settings/tokens first.

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
from huggingface_hub import HfApi
api = HfApi()
repo_id = f"{api.whoami()['name']}/imdb-sentiment-roberta"
api.create_repo(repo_id, exist_ok=True)
api.upload_folder(folder_path="models/roberta-base-512-head_tail", repo_id=repo_id)
print(f"https://huggingface.co/{repo_id}")

## 6. Try the model

In [ ]:
!python -m src.predict --model_dir models/roberta-base-512-head_tail \
    "An absolute masterpiece, I cried twice." \
    "Oh great, another two hours of explosions and zero character development."

## 7. Download the results to commit them to GitHub

In [ ]:
!zip -qr results.zip results -x "results/*_predictions.csv"
from google.colab import files
files.download("results.zip")